# DAC Latent Processor Demo

This notebook uses `utills/dac.py` to:
- Encode audio files into DAC latents
- Save latents as `.pt` files
- Load latents and optionally decode back to audio

## 1) Setup and Imports

In [ ]:
from pathlib import Path
import sys
import torch

# Add project root to sys.path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utills.dac import DACLatentProcessor

print(f"Project root: {project_root}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2) Configuration
Update the input/output directories as needed.

In [ ]:
# Configure paths and model settings
INPUT_DIR = project_root / "data" / "resynth_final"
OUTPUT_DIR = project_root / "data" / "codec" / "synth"

MODEL_TYPE = "44khz"  # '44khz', '24khz', '16khz'
N_QUANTIZERS = 9         # None for all, or an int (e.g., 9)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Input dir: {INPUT_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Model: {MODEL_TYPE}, n_quantizers={N_QUANTIZERS}, device={DEVICE}")

In [ ]:
if INPUT_DIR.exists():
    exts = {".wav", ".mp3", ".flac"}
    files = [p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in exts]
    print(f"Found {len(files)} audio files in {INPUT_DIR}")
else:
    print(f"Input dir does not exist: {INPUT_DIR}")

## 3) Initialize Processor

In [ ]:
processor = DACLatentProcessor(
    model_type=MODEL_TYPE,
    device=DEVICE,
    n_quantizers=N_QUANTIZERS,
)
print("Processor ready")

## 4) Encode a Directory (Optional)
Set `RUN_PROCESS = True` to encode and save `.pt` files.

In [ ]:
RUN_PROCESS = False

if RUN_PROCESS:
    processor.process_directory(
        input_dir=str(INPUT_DIR),
        output_dir=str(OUTPUT_DIR),
        extensions=['.wav', '.mp3', '.flac']
    )
else:
    print("Skipping processing. Set RUN_PROCESS=True to run.")

## 5) Load a Latent File
Update `SAMPLE_LATENT` to a `.pt` file in your output folder.

In [ ]:
# Pick any .pt file from the output directory
SAMPLE_LATENT = None  # Example: OUTPUT_DIR / 'your_file.pt'

if SAMPLE_LATENT is not None:
    data = processor.load_latents(str(SAMPLE_LATENT))
    z = data['z']
    print(f"Loaded latent: {SAMPLE_LATENT}")
    print(f"z shape: {z.shape}")
    print(f"sample_rate: {data['sample_rate']}")
    print(f"original_length: {data['original_length']}")
else:
    print("Set SAMPLE_LATENT to a .pt file to load.")

In [ ]:
# Optional: decode and listen
from IPython.display import Audio, display

if SAMPLE_LATENT is not None:
    audio = processor.decode_latents(z)
    # audio shape: (B, 1, T) or (B, T)
    print(f"Decoded audio shape: {audio.shape}")
    display(Audio(audio.squeeze(), rate=int(data['sample_rate'])))
else:
    print("Set SAMPLE_LATENT to decode audio.")